# DIM_CID

Diagnósticos CID-10 com o capítulo e a flag `FLG_RESPIRATORIA` (capítulo X, códigos
`J00`–`J99`), usada na pergunta sobre picos de internação por doenças respiratórias.
Códigos usados no SIH mas ausentes da tabela entram como `NAO CADASTRADO`.

**Estratégia de carga:** incremental com `MERGE` (SCD tipo 1). Registros novos recebem a surrogate key da sequence; registros existentes só são atualizados se algum atributo mudou. Reexecutar a carga não duplica linhas nem troca as chaves.

**Membro desconhecido:** `SK = -1` ("NAO INFORMADO"), usado pelas fatos quando o código de origem não existe na dimensão.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import mesclar, executar, consultar

In [ ]:
MEMBRO_DESCONHECIDO = """
MERGE INTO DIM_CID d
USING (SELECT -1 AS SK FROM DUAL) s ON (d.SK_CID = s.SK)
WHEN NOT MATCHED THEN INSERT (SK_CID, COD_CID, DESCRICAO_CID, CAPITULO_CID, FLG_RESPIRATORIA)
VALUES (-1, '-1', 'NAO INFORMADO', 'NAO INFORMADO', 'N')
"""
executar(MEMBRO_DESCONHECIDO)

In [ ]:
SQL = """
MERGE INTO DIM_CID d
USING (
    SELECT c.COD_CID,
           NVL(t.DESCRICAO_CID, 'NAO CADASTRADO - ' || c.COD_CID) AS DESCRICAO_CID,
           NVL(t.CAPITULO_CID, 'NAO INFORMADO')                   AS CAPITULO_CID,
           CASE WHEN c.COD_CID LIKE 'J%' THEN 'S' ELSE 'N' END    AS FLG_RESPIRATORIA
    FROM (SELECT COD_CID FROM SILVER_CID
          UNION SELECT COD_CID FROM SILVER_SIH WHERE COD_CID IS NOT NULL) c
    LEFT JOIN SILVER_CID t ON t.COD_CID = c.COD_CID
) s
ON (d.COD_CID = s.COD_CID)
WHEN MATCHED THEN UPDATE SET
    d.DESCRICAO_CID = s.DESCRICAO_CID, d.CAPITULO_CID = s.CAPITULO_CID,
    d.FLG_RESPIRATORIA = s.FLG_RESPIRATORIA
  WHERE DECODE(d.DESCRICAO_CID, s.DESCRICAO_CID, 0, 1) + DECODE(d.CAPITULO_CID, s.CAPITULO_CID, 0, 1)
      + DECODE(d.FLG_RESPIRATORIA, s.FLG_RESPIRATORIA, 0, 1) > 0
WHEN NOT MATCHED THEN INSERT
    (SK_CID, COD_CID, DESCRICAO_CID, CAPITULO_CID, FLG_RESPIRATORIA)
VALUES
    (SEQ_DIM_CID.NEXTVAL, s.COD_CID, s.DESCRICAO_CID, s.CAPITULO_CID, s.FLG_RESPIRATORIA)
"""

In [ ]:
mesclar("DIM_CID", SQL, origem="SILVER_CID")

In [ ]:
consultar("SELECT * FROM DIM_CID ORDER BY 1 FETCH FIRST 10 ROWS ONLY")